In [ ]:
import torch
import torch.nn as nn
from torch.nn import functional as F
import math
import time

## Configuration

In [ ]:
SPECIAL_TOKENS = ["<unk>", "<bos>", "<eos>", "<pad>"]
UNKOWN_TOKEN = "<unk>"

NUM_PROC = 8
BATCH_SIZE = 64  # How many blocks of text to feed the GPU at once

MODEL_PATH = "tinystories_model.pt"

# --- Model Hyperparameters ---
VOCAB_SIZE = 4096
CONTEXT_LENGTH = 256

# Transformer dimensions
N_EMBD = 256           # Dimensionality of the token embeddings
N_HEAD = 8             # Number of attention heads (256 / 8 = 32 dimensions per head)
N_LAYER = 4            # Number of Transformer blocks
DROPOUT = 0.1          # Dropout rate to prevent overfitting

# --- Training Configurations ---
EPOCHS = 2
LEARNING_RATE = 5e-4

# Check if GPU is available
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

# Transformer Architecture

## Self Attention Mechanism

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self):
        super().__init__()
        # Key, Query, and Value projections all bundled into one matrix for speed
        self.c_attn = nn.Linear(N_EMBD, 3 * N_EMBD, bias=False)

        # Output projection
        self.c_proj = nn.Linear(N_EMBD, N_EMBD, bias=False)

        # Dropouts: Standard regularization techniques to randomly turn off a percentage of neurons during training 
        # to prevent the model from memorizing the data (overfitting).
        self.attn_dropout = nn.Dropout(DROPOUT)
        self.resid_dropout = nn.Dropout(DROPOUT)
        
        # Causal mask to ensure attention is only applied to the left (past tokens)
        # We register it as a buffer so PyTorch moves it to the GPU but doesn't train it
        self.register_buffer("bias", torch.tril(torch.ones(CONTEXT_LENGTH, CONTEXT_LENGTH))
                                     .view(1, 1, CONTEXT_LENGTH, CONTEXT_LENGTH))

    def forward(self, x):
        B, T, C = x.size() # Batch, Time (Context Length), Channels (Embedding Dim)

        # Calculate Query, Key, Values
        qkv = self.c_attn(x) # [B, T, 3 * N_EMBD]
        q, k, v = qkv.split(N_EMBD, dim=2) # 3 x [B, T, N_EMBD]

        # Reshape for multi-head attention
        # [B, T, N_EMBD] -> [B, num_heads, T, head_size]
        k = k.view(B, T, N_HEAD, C // N_HEAD).transpose(1, 2)
        q = q.view(B, T, N_HEAD, C // N_HEAD).transpose(1, 2)
        v = v.view(B, T, N_HEAD, C // N_HEAD).transpose(1, 2)

        # Scaled Dot-Product Attention: Softmax(Q K^T / sqrt(d_k)) V
        att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
        
        # Apply the causal mask (replace upper triangle with -infinity so softmax makes them 0)
        att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)

        # Multiply by Values
        y = att @ v 
        
        # Re-assemble all head outputs side by side
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        
        # Final output projection
        y = self.resid_dropout(self.c_proj(y))
        return y

## Multi-Layer Perceptron (MLP)

In [ ]:
class FeedForward(nn.Module):
    def __init__(self):
        super().__init__()
        # In Transformers, the hidden layer is typically 4x larger than the embedding
        self.net = nn.Sequential(
            nn.Linear(N_EMBD, 4 * N_EMBD, bias=False),
            nn.GELU(),
            nn.Linear(4 * N_EMBD, N_EMBD, bias=False),
            nn.Dropout(DROPOUT),
        )

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln_1 = nn.LayerNorm(N_EMBD)
        self.attn = CausalSelfAttention()
        self.ln_2 = nn.LayerNorm(N_EMBD)
        self.mlp = FeedForward()

    def forward(self, x):
        # The x + ... represents residual (skip) connections. 
        # These are crucial for helping deep networks learn.
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x

## Full language model

In [ ]:
class TinyStoriesModel(nn.Module):
    def __init__(self):
        super().__init__()
        
        # Token Embeddings (Translates token IDs to vectors)
        self.token_embedding_table = nn.Embedding(VOCAB_SIZE, N_EMBD)
        # Positional Embeddings (Tells the model where the word is in the sequence)
        self.position_embedding_table = nn.Embedding(CONTEXT_LENGTH, N_EMBD)
        
        # The main body: A sequence of Transformer blocks
        self.blocks = nn.Sequential(*[Block() for _ in range(N_LAYER)])
        
        # Final normalization and linear projection back to Vocabulary Size
        self.ln_f = nn.LayerNorm(N_EMBD)
        self.lm_head = nn.Linear(N_EMBD, VOCAB_SIZE, bias=False)

        # Enforce weight tying (weight sharing) between token embeddings and the LM head.
        self.token_embedding_table.weight = self.lm_head.weight

    def forward(self, idx, targets=None):
        B, T = idx.size()
        
        # Look up embeddings
        tok_emb = self.token_embedding_table(idx) # (B, T, N_EMBD)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T, N_EMBD)
        
        # Add them together
        x = tok_emb + pos_emb
        
        # Pass through the transformer
        x = self.blocks(x)
        x = self.ln_f(x)
        
        # Get raw predictions (logits) for the next tokens
        logits = self.lm_head(x) # (B, T, VOCAB_SIZE)
        
        loss = None
        if targets is not None:
            # Shift so that tokens < n predict n
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = targets[..., 1:].contiguous()
            
            # Flatten the tokens
            loss = F.cross_entropy(
                shift_logits.view(-1, shift_logits.size(-1)), 
                shift_labels.view(-1)
            )
            
        return logits, loss

# Training

## Load Dataset

In [ ]:
from datasets import load_dataset

# Load the TinyStories dataset
print("Downloading TinyStories dataset...")
dataset = load_dataset("roneneldan/TinyStories")

# Let's inspect what we just downloaded
print(dataset)

# Print a sample story to see what it looks like
print("\n--- Sample Story ---")
print(dataset["train"][0]["text"])

## Generate vocabulary and create tokenizer

In [ ]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# Instantiate the BPE model backbone
tokenizer = Tokenizer(BPE(unk_token=UNKOWN_TOKEN))

# Pre-tokenizer to split text into words by whitespace
tokenizer.pre_tokenizer = Whitespace()

# Setup the trainer
trainer = BpeTrainer(
    vocab_size=VOCAB_SIZE,                          
    special_tokens=SPECIAL_TOKENS
)

# Create a generator to stream the dataset in chunks
# This feeds the text to the tokenizer 10,000 stories at a time
def batch_iterator(batch_size=10_000):
    for i in range(0, len(dataset["train"]), batch_size):
        yield dataset["train"][i : i + batch_size]["text"]

# Train the tokenizer using the iterator instead of a text file
print("Training BPE tokenizer from dataset (this may take a minute)...")
tokenizer.train_from_iterator(batch_iterator(), trainer=trainer)

# Save the trained tokenizer configuration to disk
tokenizer.save("tinystories_bpe.json")
print("BPE Tokenizer successfully trained and saved!")

### Post processing for tokenizer (not necessary, just for debuging)

In [ ]:
from tokenizers.processors import TemplateProcessing

# Load your tokenizer just to be sure
tokenizer = Tokenizer.from_file("tinystories_bpe.json")

# Configure the tokenizer to always wrap text in <bos> and <eos>
tokenizer.post_processor = TemplateProcessing(
    single="<bos> $A <eos>",
    special_tokens=[
        ("<bos>", tokenizer.token_to_id("<bos>")),
        ("<eos>", tokenizer.token_to_id("<eos>")),
    ],
)

# Test it out!
test_phrase = "Lily found a little bird."
encoded = tokenizer.encode(test_phrase)

print("Tokens:", encoded.tokens)
print("IDs   :", encoded.ids)

## Tokenize Dataset

In [ ]:
def tokenize_function(examples):
    from tokenizers import Tokenizer
    from tokenizers.processors import TemplateProcessing
    
    # Every background process loads its own copy of the tokenizer
    worker_tokenizer = Tokenizer.from_file("tinystories_bpe.json")
    
    # Re-apply the post-processor to make sure <bos> and <eos> are added by the workers
    worker_tokenizer.post_processor = TemplateProcessing(
        single="<bos> $A <eos>",
        special_tokens=[
            ("<bos>", worker_tokenizer.token_to_id("<bos>")),
            ("<eos>", worker_tokenizer.token_to_id("<eos>")),
        ],
    )
    
    # Process the current batch of stories
    outputs = [worker_tokenizer.encode(text).ids for text in examples["text"]]
    return {"input_ids": outputs}

print("Tokenizing the entire dataset using parallel processing...")
tokenized_datasets = dataset.map(
    tokenize_function,
    batched=True,
    num_proc=NUM_PROC,
    remove_columns=["text"] # Drops the raw strings to save RAM
)

print(tokenized_datasets)

## Group Tokens

In [ ]:
def group_texts(examples, context_length):
    # Concatenate all lists of IDs together into one massive list
    concatenated_ids = sum(examples["input_ids"], [])
    total_length = len(concatenated_ids)
    
    # Chop off the tiny remainder at the very end so it divides perfectly
    total_length = (total_length // context_length) * context_length
    
    # Split the massive list into chunks of context_length
    result = {
        "input_ids": [
            concatenated_ids[i : i + context_length]
            for i in range(0, total_length, context_length)
        ]
    }
    
    # For language modeling, the labels are the exact same as the inputs
    # (The transformer will automatically shift them by 1 internally to predict the "next" token)
    result["labels"] = result["input_ids"].copy()
    return result

print("Chunking tokens into fixed sizes...")
lm_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    num_proc=NUM_PROC,
    fn_kwargs={"context_length": CONTEXT_LENGTH} 
)

print(lm_datasets)

## PyTorch data loaders

In [ ]:
from torch.utils.data import DataLoader

# Convert the dataset to return PyTorch tensors instead of standard Python lists
lm_datasets.set_format(type="torch", columns=["input_ids", "labels"])

# Create the training loader (shuffled so the model doesn't memorize the order)
train_dataloader = DataLoader(
    lm_datasets["train"], 
    shuffle=True, 
    batch_size=BATCH_SIZE
)

# Create the validation loader (doesn't need to be shuffled)
val_dataloader = DataLoader(
    lm_datasets["validation"], 
    shuffle=False, 
    batch_size=BATCH_SIZE
)

# Verify the tensor shapes of the first batch
batch = next(iter(train_dataloader))
print("Batch Input IDs shape :", batch["input_ids"].shape)
print("Batch Labels shape    :", batch["labels"].shape)

## Test

In [ ]:
# --- Validate architecture initialization and forward pass ---
model = TinyStoriesModel().to(device)

# Print total parameters
num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model initialized! Total Parameters: {num_params:,}")

# Do a dummy forward pass with the batch we loaded earlier
xb = batch["input_ids"].to(device)
yb = batch["labels"].to(device)

logits, loss = model(xb, yb)
print(f"Logits shape: {logits.shape}")
print(f"Initial Loss: {loss.item():.4f}")

## Train

In [ ]:
# Initialize optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

print("Starting training loop...")
for epoch in range(EPOCHS):
    model.train()
    total_train_loss = 0
    start_time = time.time()
    
    for step, batch in enumerate(train_dataloader):
        xb = batch["input_ids"].to(device)
        yb = batch["labels"].to(device)
        
        # Forward pass & loss calculation
        logits, loss = model(xb, yb)
        
        # Backward pass
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        
        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        total_train_loss += loss.item()
        
        if step % 200 == 0:
            print(f"Epoch {epoch+1}/{EPOCHS} | Step {step}/{len(train_dataloader)} | Current Loss: {loss.item():.4f}")
            
    # --- Validation Phase ---
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch in val_dataloader:
            xb = batch["input_ids"].to(device)
            yb = batch["labels"].to(device)
            _, loss = model(xb, yb)
            total_val_loss += loss.item()
            
    avg_train_loss = total_train_loss / len(train_dataloader)
    avg_val_loss = total_val_loss / len(val_dataloader)
    elapsed_time = time.time() - start_time
    
    print(f"\Epoch {epoch+1} Complete ({elapsed_time:.2f}s)")
    print(f"Avg Train Loss: {avg_train_loss:.4f} | Avg Val Loss: {avg_val_loss:.4f}\n")

# Save the trained model parameters
torch.save(model.state_dict(), MODEL_PATH)
print(f"Model weights successfully saved to '{MODEL_PATH}'")